# บทเรียน 11 - โปรโตคอล เอเจนต์ต่อเอเจนต์ (A2A)


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## โปรโตคอล A2A คืออะไร?

โปรโตคอล **Agent-to-Agent (A2A)** เป็นมาตรฐานเปิดที่ช่วยให้เอเจนต์ AI สามารถสื่อสาร,
ค้นหาและทำงานร่วมกัน — แม้จะถูกสร้างบนเฟรมเวิร์กที่แตกต่างกันหรือโฮสต์
โดยบริการที่แตกต่างกัน.

Key concepts:

- **Discovery** – เอเจนต์เผยแพร่ *Agent Card* ที่อธิบายความสามารถของพวกเขา ทำให้
  เอเจนต์อื่น (หรือตัวจัดการการประสานงาน) ค้นหาผู้เชี่ยวชาญที่เหมาะสมสำหรับงานได้ง่าย.
- **Message Passing** – เอเจนต์แลกเปลี่ยนข้อความที่มีโครงสร้างผ่านโปรโตคอลร่วม ดังนั้นคำขอจาก
  เอเจนต์หนึ่งสามารถถูกเข้าใจและดำเนินการได้โดยอีกเอเจนต์หนึ่งโดยไม่ขึ้นกับการออกแบบภายใน
  ของมัน.
- **Task Lifecycle** – A2A ระบุสถานะต่างๆ เช่น *submitted*, *working*, *completed*, และ
  *failed*, ทำให้ตัวจัดการกระบวนการมองเห็นได้เต็มที่ว่าหน้าที่ถูกมอบหมายกำลังดำเนินไปอย่างไร.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
เข้าสู่เวิร์กโฟลว์ที่แต่ละเอเจนต์นำความเชี่ยวชาญของตนมามีส่วนช่วยและส่งผลลัพธ์ต่อเอเจนต์ถัดไป.


## การสร้างตัวแทนท่องเที่ยวเฉพาะทาง


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ความร่วมมือของหลายเอเจนต์ผ่านเวิร์กโฟลว์

เราเชื่อมต่อเอเจนต์ทั้งสามเข้าเป็นเวิร์กโฟลว์แบบลำดับที่สะท้อนการส่งข้อความแบบ A2A:

1. **CurrencyExchangeAgent** รับคำขอจากผู้ใช้และให้คำแนะนำด้านอัตราแลกเปลี่ยน.
2. **ActivityPlannerAgent** รับบริบทที่ได้รับการเสริมและเพิ่มคำแนะนำกิจกรรม.
3. **TravelManagerAgent** สังเคราะห์ข้อมูลทั้งสองเป็นสรุปการเดินทางขั้นสุดท้าย.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ทำความเข้าใจ A2A ในการใช้งานจริง

ในสภาพแวดล้อมการใช้งานจริง โปรโตคอล A2A จะเปิดประตูสู่กรณีการใช้งานข้ามบริการที่ทรงพลัง:

| ความสามารถ | คำอธิบาย |
|---|---|
| **การทำงานร่วมข้ามเฟรมเวิร์ก** | เอเจนต์ที่สร้างด้วยเฟรมเวิร์กหนึ่งสามารถมอบหมายงานให้กับเอเจนต์ที่สร้างด้วยเฟรมเวิร์กอื่นๆ ที่สอดคล้องกับ A2A ได้ ทำให้เกิดความสามารถในการทำงานร่วมกันข้ามองค์กรอย่างแท้จริง. |
| **ขอบเขตของบริการ** | เอเจนต์สามารถอยู่ในไมโครเซอร์วิสต่างกัน ภูมิภาคคลาวด์ต่างกัน หรือแม้แต่ในองค์กรต่างๆ ในขณะที่ยังคงทำงานร่วมกันได้อย่างราบรื่น. |
| **การค้นพบแบบไดนามิก** | ออเคสเตรเตอร์สามารถสอบถามรีจิสทรีของ Agent Card ขณะรันไทม์เพื่อค้นหาผู้เชี่ยวชาญที่เหมาะสมที่สุดสำหรับงานย่อยที่กำหนด. |
| **การสตรีมและการแจ้งเตือนแบบพุช** | A2A รองรับ Server-Sent Events (SSE) สำหรับการอัปเดตความคืบหน้าแบบเรียลไทม์ และการแจ้งเตือนแบบพุชสำหรับงานที่ใช้เวลานาน. |

เวิร์กโฟลว์ที่เราได้สร้างไว้ข้างต้นเป็นเวอร์ชันที่เรียบง่ายและทำงานภายในกระบวนการของรูปแบบนี้ ในการปรับใช้
จริง แต่ละเอเจนต์จะเปิดเผย HTTP endpoint เผยแพร่ Agent Card และสื่อสาร
ผ่านโปรโตคอล A2A JSON-RPC.


## สรุป

ในบทเรียนนี้คุณได้เรียนรู้:

1. **โปรโตคอล A2A คืออะไร** — มาตรฐานเปิดสำหรับการค้นพบ การส่งข้อความ,
   และการจัดการงาน.
2. **วิธีสร้างตัวแทนเฉพาะทาง** — ตัวแทนแลกเปลี่ยนสกุลเงิน, ตัวแทนวางแผนกิจกรรม, และตัวประสานผู้จัดการการเดินทาง.
3. **วิธีเชื่อมตัวแทนเข้ากับเวิร์กโฟลว์** — ใช้ `WorkflowBuilder` เพื่อจำลองการส่งข้อความแบบลำดับ
   ระหว่างตัวแทน.
4. **วิธีที่ A2A ทำงานในการใช้งานจริง** — อำนวยความสะดวกให้เกิดความร่วมมือข้ามเฟรมเวิร์ก, ข้ามบริการ พร้อมการค้นพบแบบไดนามิกและการอัปเดตแบบสตรีมมิง.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารฉบับนี้ถูกแปลโดยใช้บริการแปลภาษาด้วยปัญญาประดิษฐ์ Co-op Translator (https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้การแปลมีความถูกต้อง โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อน เอกสารต้นฉบับในภาษาดั้งเดิมควรถูกพิจารณาเป็นแหล่งข้อมูลที่มีอำนาจ หากเป็นข้อมูลที่มีความสำคัญ ขอแนะนำให้ใช้การแปลโดยนักแปลมืออาชีพ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใด ๆ ที่เกิดจากการใช้การแปลนี้.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
